In [ ]:
import sys
import numpy as np
import pygame
import gymnasium as gym

from gymnasium.wrappers import RecordVideo


Python: c:\Users\gamin\OneDrive\Desktop\gam3a\fall 2026\rl\labs\lab 2\RL-lab2\.venv\Scripts\python.exe
NumPy: 2.5.4
Pygame: 2.6.1
Gymnasium: 1.4.0
RecordVideo: <class 'gymnasium.wrappers.rendering.RecordVideo'>
All core imports successful!


In [ ]:
import sys
import numpy as np

if "./gym_env" not in sys.path:
    sys.path.insert(0, "./gym_env")

from gym_env.envs.grid_world import GridWorldEnv, Actions

def get_step_outcome(env, state, action):
    """Computes (next_state, reward, terminated) for a given (x, y) state and action."""
    pos = np.array(state, dtype=int)
    # Terminal states (target or bad cells) have 0 future transitions
    if (
        np.array_equal(pos, env._target_location)
        or np.array_equal(pos, env._bad1_location)
        or np.array_equal(pos, env._bad2_location)
    ):
        return tuple(pos), 0.0, True

    direction = env._action_to_direction[action]
    next_pos = np.clip(pos + direction, 0, env.size - 1)

    if np.array_equal(next_pos, env._target_location):
        return tuple(next_pos), 1.0, True
    elif np.array_equal(next_pos, env._bad1_location) or np.array_equal(next_pos, env._bad2_location):
        return tuple(next_pos), -1.0, True
    return tuple(next_pos), 0.0, False


def policy_iteration(env, gamma=0.9, theta=1e-6):
    policy = np.zeros((env.size, env.size), dtype=int)
    V = np.zeros((env.size, env.size), dtype=float)
    iterations = 0

    while True:
        iterations += 1
        # 1. Policy Evaluation
        while True:
            delta = 0.0
            for x in range(env.size):
                for y in range(env.size):
                    v_old = V[x, y]
                    (nx, ny), reward, terminated = get_step_outcome(env, (x, y), policy[x, y])
                    V[x, y] = reward + (0.0 if terminated else gamma * V[nx, ny])
                    delta = max(delta, abs(v_old - V[x, y]))
            if delta < theta:
                break

        # 2. Policy Improvement
        policy_stable = True
        for x in range(env.size):
            for y in range(env.size):
                old_action = policy[x, y]
                q_values = np.zeros(env.action_space.n)
                for a in range(env.action_space.n):
                    (nx, ny), reward, terminated = get_step_outcome(env, (x, y), a)
                    q_values[a] = reward + (0.0 if terminated else gamma * V[nx, ny])

                best_action = int(np.argmax(q_values))
                policy[x, y] = best_action
                if best_action != old_action and not np.isclose(q_values[best_action], q_values[old_action]):
                    policy_stable = False

        if policy_stable:
            break

    return policy, V, iterations

In [13]:
# Cell 3: Run Policy Iteration and Record Video
from gymnasium.wrappers import RecordVideo

env = GridWorldEnv(render_mode="rgb_array", size=5)
video_env = RecordVideo(
    env,
    video_folder="./videos",
    name_prefix="grid_maze_agent",
    episode_trigger=lambda ep: True,
)

obs, info = video_env.reset(seed=42)
policy, V, iters = policy_iteration(video_env.unwrapped, gamma=0.9)
print(f"Policy Iteration converged in {iters} iterations.")

terminated = False
steps = 0
while not terminated and steps < 50:
    agent_x, agent_y = obs["agent"]
    action = int(policy[agent_x, agent_y])
    obs, reward, terminated, truncated, info = video_env.step(action)
    steps += 1

video_env.close()
print(f"Episode finished in {steps} steps with final reward {reward}. Video saved to ./videos/")

Policy Iteration converged in 4 iterations.
Episode finished in 4 steps with final reward 1. Video saved to ./videos/


c:\Users\gamin\OneDrive\Desktop\gam3a\fall 2026\rl\labs\lab 2\RL-lab2\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:443: UserWarning: WARN: Unable to save last video! Did you call close()?
  logger.warn("Unable to save last video! Did you call close()?")


In [14]:
from gymnasium.wrappers import RecordVideo

video_env = GridWorldEnv(render_mode="rgb_array", size=5)
video_env = RecordVideo(
    video_env,
    video_folder="./videos",
    name_prefix="grid_maze_policy_iteration",
    episode_trigger=lambda episode_id: True,
)

num_episodes = 5
for ep in range(num_episodes):
    obs, info = video_env.reset(seed=100 + ep)
    
    # Unpack 3 values matching policy_iteration()
    policy, V, iters = policy_iteration(video_env.unwrapped, gamma=0.9)

    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated):
        agent_x, agent_y = obs["agent"]
        action = int(policy[agent_x, agent_y])
        obs, reward, terminated, truncated, info = video_env.step(action)
        total_reward += reward
        steps += 1

    print(f"Episode {ep + 1}: Converged in {iters} iterations | Steps: {steps} | Total Reward = {total_reward}")

video_env.close()
print("Videos successfully saved to ./videos/")

Episode 1: Converged in 7 iterations | Steps: 5 | Total Reward = 1.0
Episode 2: Converged in 5 iterations | Steps: 5 | Total Reward = 1.0
Episode 3: Converged in 5 iterations | Steps: 2 | Total Reward = 1.0
Episode 4: Converged in 8 iterations | Steps: 2 | Total Reward = 1.0
Episode 5: Converged in 7 iterations | Steps: 3 | Total Reward = 1.0
Videos successfully saved to ./videos/
